# Fig. 4: trait and functional-diversity effects on GPP within aridity classes

**Purpose.** Plot, for each aridity class (Arid, Semi-arid, Dry sub-humid, Sub-humid, Humid),
the coefficient (with 95% CI) of each trait PC (PC1-PC3) and of each functional-diversity
metric (functional richness, divergence and dissimilarity at the alpha / gamma / tau scales)
from the per-class linear mixed-effects models. Filled points are significant (p < 0.05),
open points are not.

**Inputs** (written by `03_evaluate_models.Rmd`):
- `results/LME_GPP_LAI_T_P_AI_class_trait_ecoprovince_SA_results.csv`
- `results/LME_GPP_LAI_T_P_AI_class_FD_ecoprovince_SA_results.csv`

**Output:** `results/fig4_aridity_classes.png`

**Run order:** last step in `aridity_classes/` (after `01` -> `02` -> `03`), with the working
directory set to `aridity_classes/`.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_style('ticks')
sns.set_context('paper')
plt.rcParams['font.family'] = ['Helvetica', 'Arial', 'DejaVu Sans']

os.makedirs('results', exist_ok=True)


# Functional composition model

In [ ]:
functional_composition = pd.read_csv('results/LME_GPP_LAI_T_P_AI_class_trait_ecoprovince_SA_results.csv')


# Functional diversity model

In [ ]:
functional_diversity = pd.read_csv('results/LME_GPP_LAI_T_P_AI_class_FD_ecoprovince_SA_results.csv')

# Plot results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(7.5, 5.5), layout="constrained")
axes = axes.flatten()

# Background colour for each aridity class
bg_colors = ['#8E4E21', '#ECC97D', '#ECF4DF', '#83CEC4', '#10645A']

# Shade the aridity-class columns before drawing the points
for ax in axes:
    for i, category in enumerate(functional_composition['aridity_index_class'].unique()):
        ax.axvspan(i - 0.5, i + 0.5, color=bg_colors[i], zorder=0, alpha=0.4, linewidth=0)

point_color = '#424856'
errorbar_color = '#8F97A4'
# plot CI bar

PC_list = ['all_PC1', 'all_PC2', 'all_PC3']
for i, PC in enumerate(PC_list):
    yerr = [functional_composition.loc[functional_composition['trait'] == PC, 'coef_trait'] -
            functional_composition.loc[functional_composition['trait'] == PC, 'coef_trait_CI_lower'],
            functional_composition.loc[functional_composition['trait'] == PC, 'coef_trait_CI_upper'] -
            functional_composition.loc[functional_composition['trait'] == PC, 'coef_trait']]
    axes[i].errorbar(x=functional_composition.loc[functional_composition['trait'] == PC, 'aridity_index_class'],
                     y=functional_composition.loc[functional_composition['trait'] == PC, 'coef_trait'],
                     yerr=yerr, fmt='o', color=errorbar_color, markersize=0, linewidth=1, capsize=2)
    index_insignificant = (functional_composition['trait'] == PC) & (
            functional_composition['p_value_trait'] >= 0.05)
    index_significant = (functional_composition['trait'] == PC) & (
            functional_composition['p_value_trait'] < 0.05)
    axes[i].scatter(functional_composition.loc[index_significant, 'aridity_index_class'],
                     functional_composition.loc[index_significant, 'coef_trait'],
                     color=point_color, s=13, zorder=5)
    axes[i].scatter(functional_composition.loc[index_insignificant, 'aridity_index_class'],
                        functional_composition.loc[index_insignificant, 'coef_trait'],
                        color=point_color, s=13, zorder=5, facecolors='white')

    axes[i].plot(functional_composition.loc[functional_composition['trait'] == PC, 'aridity_index_class'],
                 functional_composition.loc[functional_composition['trait'] == PC, 'coef_trait'],
                 color=point_color, linewidth=1.5, zorder=4)


FD_list = [{'FRic': ['alpha', 'gamma', 'tau']}, {'FDiv': ['alpha', 'gamma', 'tau']},
           {'Fbeta': ['alpha_to_gamma', 'gamma_to_tau']}]

color_dict = {'alpha': '#FF6347', 'gamma': '#F4A460', 'tau': '#FFD700',
              'alpha_to_gamma': '#6A5ACD', 'gamma_to_tau': '#469eb4'}
for FD_metric in FD_list:
    for scale_list in FD_metric.values():
        for scale in scale_list:
            FD = list(FD_metric.keys())[0] + '_' + scale
            # plot CI bar
            yerr = [functional_diversity.loc[functional_diversity['FD'] == FD, 'coef_FD'] -
                    functional_diversity.loc[
                        functional_diversity['FD'] == FD, 'coef_FD_CI_lower'],
                    functional_diversity.loc[
                        functional_diversity['FD'] == FD, 'coef_FD_CI_upper'] -
                    functional_diversity.loc[functional_diversity['FD'] == FD, 'coef_FD']]
            axes[FD_list.index(FD_metric) + 3].errorbar(
                x=functional_diversity.loc[
                    functional_diversity['FD'] == FD, 'aridity_index_class'],
                y=functional_diversity.loc[functional_diversity['FD'] == FD, 'coef_FD'],
                yerr=yerr, fmt='o', color=errorbar_color, markersize=0, linewidth=1, capsize=2)
            index_significant = (functional_diversity['FD'] == FD) & (
                    functional_diversity['p_value_FD'] < 0.05)
            index_insignificant = (functional_diversity['FD'] == FD) & (
                    functional_diversity['p_value_FD'] >= 0.05)
            axes[FD_list.index(FD_metric) + 3].scatter(
                functional_diversity.loc[index_significant, 'aridity_index_class'],
                functional_diversity.loc[index_significant, 'coef_FD'],
                color=color_dict[scale], s=13, zorder=5)
            axes[FD_list.index(FD_metric) + 3].scatter(
                functional_diversity.loc[index_insignificant, 'aridity_index_class'],
                functional_diversity.loc[index_insignificant, 'coef_FD'],
                color=color_dict[scale], s=13, zorder=5, facecolors='white')

            axes[FD_list.index(FD_metric) + 3].plot(functional_diversity.loc[
                                                        functional_diversity['FD'] == FD, 'aridity_index_class'],
                                                    functional_diversity.loc[
                                                        functional_diversity['FD'] == FD, 'coef_FD'],
                                                    color=color_dict[scale], linewidth=1.5, zorder=4)

# axes[0].set_ylim(0, 1)
# axes[5].set_ylim(0, 0.8)
# remove y label and axis label except for the first column
for i, ax in enumerate(axes):
    ax.set_xlim(-0.5, 4.5)
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel('Effect size\n(coefficient)')
    else:
        ax.set_ylabel('')
    if ax.get_subplotspec().is_last_row():
        ax.set_xlabel('')
        # rotate x axis label
        for label in ax.get_xticklabels():
            label.set_rotation(90)
            label.set_ha('center')
    else:
        ax.set_xlabel('')
        ax.set_xticklabels('')
        ax.set_xticks([])
    # set axex[0] and axes[1] to have the same y axis

    if i in [1, 2, 4, 5]:
        ax.set_yticklabels('')
        ax.set_yticks([])
    if i in [0, 1, 2]:
        ax.set_ylim(-0.4, 0.7)
    else:
        ax.set_ylim(-0.2, 0.6)
    # add y = 0 line, dashed grey line
    ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.7)

    # remove right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    # set axis line width
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)

scale_dict = {'alpha': r'$\bar{\alpha}$', 'gamma': r'$\bar{\gamma}$', 'tau': r'$\tau$',
              'alpha_to_gamma': r'$\bar{\beta}_{\alpha \rightarrow \gamma}$',
              'gamma_to_tau': r'$\beta_{\gamma \rightarrow \tau}$'}
# add legend of axes[2]
for scale in ['alpha', 'gamma', 'tau']:
    axes[4].plot([], [], color=color_dict[scale], label=str(scale_dict[scale]), linewidth=1.5)
leg = axes[4].legend(loc='upper right', bbox_to_anchor=(0.13, 1.0),
                     fontsize='small', frameon=True, handletextpad=0.5, borderpad=0.3)
leg.set_in_layout(False)
# add legend of axes[4]
for scale in ['alpha_to_gamma', 'gamma_to_tau']:
    axes[5].plot([], [], color=color_dict[scale], label=str(scale_dict[scale]), linewidth=1.5)
axes[5].legend(loc='upper right', bbox_to_anchor=(1.0, 1.0),
               fontsize='small', frameon=True, handletextpad=0.5, borderpad=0.3)

# add legend of non-significant and significant
axes[0].scatter([], [], color='black', s=13, label='$p < 0.05$')
axes[0].scatter([], [], color='black', s=13, facecolors='white', label=r'$p \geq 0.05$')
axes[0].legend(loc='upper right', bbox_to_anchor=(1, 1), title_fontsize='small',
               fontsize='small', frameon=True, handletextpad=0., borderpad=0.2)

# add title
for ax, title in zip(axes,
                     ['PC1', 'PC2', 'PC3', 'Functional richness', 'Functional divergence', 'Functional dissimilarity']):
    ax.set_title(title)
# add title number
for ax, title in zip(axes, ['a', 'b', 'c', 'd', 'e', 'f']):
    ax.set_title(title, loc='left', fontweight='bold')

fig.get_layout_engine().set(w_pad=0 / 72, h_pad=0 / 72, hspace=0., wspace=0.)
# add space of axes[2] y axis label
plt.savefig('results/fig4_aridity_classes.png', dpi=800, bbox_inches='tight')

plt.show()